# Capstone — Dự Đoán App Có Phổ Biến Hay Không
Một logistic regression dự đoán một app trên Google Play Store có **phổ biến (Popular)** hay không, dựa trên các đặc điểm cơ bản của app — xây dựng từ đầu tới cuối: load → split → fit → predict → score.

**Dataset:** `googleplaystore_cleaned.csv`. Đặt file này cùng thư mục với notebook.

## Bước 1 — Load Dữ Liệu, Chọn Features và Target

In [1]:
import pandas as pd

df = pd.read_csv("googleplaystore_cleaned.csv")

# Bỏ các dòng thiếu Size_MB (feature quan trọng, không nên fill giả)
df = df.dropna(subset=["Size_MB"])

# Target: app "phổ biến" nếu số lượt cài đặt >= 500,000 (ngưỡng median của dữ liệu)
# 1 = Popular, 0 = Not Popular
y = (df["Installs_Num"] >= 500_000).astype(int)

features = ["Category", "Size_MB", "Price_Num", "Type", "Content Rating"]
X = df[features]

X.head()

,Category,Size_MB,Price_Num,Type,Content Rating
0,ART_AND_DESIGN,19.0,0.0,Free,Everyone
1,ART_AND_DESIGN,14.0,0.0,Free,Everyone
2,ART_AND_DESIGN,8.7,0.0,Free,Everyone
3,ART_AND_DESIGN,25.0,0.0,Free,Teen
4,ART_AND_DESIGN,2.8,0.0,Free,Everyone


`X` là những gì model được phép nhìn thấy. `y` là đáp án đúng (1 = app phổ biến, 0 = app không phổ biến).

Ta không dùng `Reviews` hay `Rating` làm feature — hai cột này gần như luôn cao khi app đã phổ biến, nên nếu đưa vào model sẽ bị **data leakage** (model "ăn gian" bằng cách nhìn thấy hệ quả của việc phổ biến, thay vì học nguyên nhân).

## Bước 2 — Encode Text Columns, Rồi Split

In [2]:
X = pd.get_dummies(X, columns=["Category", "Type", "Content Rating"], drop_first=True)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

X_train.shape, X_test.shape

((5939, 40), (1485, 40))

`get_dummies` biến các cột chữ (Category, Type, Content Rating) thành các cột 0/1. `train_test_split` giữ lại 20% app để chấm điểm model một cách khách quan ở cuối. `stratify=y` giúp giữ tỉ lệ Popular/Not Popular cân bằng ở cả hai tập.

## Bước 3 — Fit (Train) Logistic Regression

In [3]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

Ta dùng **Logistic Regression** vì đây là model đơn giản, nhanh, dễ giải thích — phù hợp làm điểm khởi đầu (baseline) trước khi thử các model phức tạp hơn.

`max_iter=1000` đảm bảo thuật toán có đủ vòng lặp để hội tụ (mặc định 100 vòng đôi khi không đủ với dữ liệu nhiều cột).

## Bước 4 — Predict Trên Tập Test

In [4]:
y_pred = model.predict(X_test)

y_pred[:10]

array([0, 0, 0, 1, 1, 0, 0, 0, 0, 0])

`y_pred` là dự đoán của model cho 1,485 app trong tập test — mỗi app được gán **1 (Popular)** hoặc **0 (Not Popular)**, dựa trên những gì model học được ở Bước 3. Model chưa từng thấy các app này trong lúc train, nên đây là bài kiểm tra khách quan.

## Bước 5 — Score (Đánh Giá Bằng Confusion Matrix)

In [5]:
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(y_test, y_pred)
print(cm)
print()
print(classification_report(y_test, y_pred, target_names=["Not Popular", "Popular"]))

[[651 169]
 [254 411]]

              precision    recall  f1-score   support

 Not Popular       0.72      0.79      0.75       820
     Popular       0.71      0.62      0.66       665

    accuracy                           0.72      1485
   macro avg       0.71      0.71      0.71      1485
weighted avg       0.71      0.72      0.71      1485



### Kết quả này nghĩa là gì (nói theo cách dễ hiểu)

Model đúng **72%** số lần trên 1,485 app chưa từng thấy trước đó. Nghe có vẻ ổn, nhưng con số quan trọng hơn nằm ở **Popular row**: model chỉ bắt được **62%** app thực sự phổ biến — nghĩa là cứ 10 app hot thì có gần 4 app bị model bỏ sót.

### Đọc confusion matrix, từng số một (nói theo cách dễ hiểu)

```
              Dự đoán: Not Popular   Dự đoán: Popular
Thực: Not Popular       651 (TN)           169 (FP)
Thực: Popular           254 (FN)           411 (TP)
```

- **651 (True Negative)** — app thực sự không hot, model đoán đúng là không hot
- **169 (False Positive)** — app thực sự không hot, nhưng model đoán nhầm là hot ("báo động giả")
- **254 (False Negative)** — app thực sự hot, nhưng model **bỏ sót**, đoán nhầm là không hot
- **411 (True Positive)** — app thực sự hot, model bắt đúng

Trong số 665 app thực sự phổ biến (254 + 411), model chỉ bắt được 411:

Recall = bắt đúng ÷ tổng số thực sự phổ biến = 411 ÷ 665 = **0.62 = 62%**

Hãy tưởng tượng một cái lưới bắt cá thả trên một cái hồ có 665 con cá chắc chắn sẽ bơi qua. Lưới chỉ vớt được 411 con — **62%**. 254 con còn lại lọt qua lỗ lưới mà không bị bắt.

Đó là điểm cần lưu ý: dù nhìn tổng thể model có vẻ ổn (72% accuracy), nhưng ở đúng việc nó cần làm — phát hiện app tiềm năng phổ biến — nó bỏ sót gần 4 trong 10 app đáng lẽ phải bắt được.

### Recall vs. Precision — khác nhau ở đâu?

Cả hai đều nhìn vào dự đoán "Popular", nhưng đặt câu hỏi ngược nhau.

**Recall — "Trong số app THỰC SỰ phổ biến, ta bắt được bao nhiêu?"**
- Nhìn vào sự thật (665 app thực sự hot) và hỏi: ta có tìm ra chúng không?
- Công thức: bắt đúng ÷ tổng số thực sự hot = 411 ÷ 665 = **62%**
- Recall thấp nghĩa là nhiều app hot thực sự bị lọt qua, không được phát hiện.

**Precision — "Trong số app model GẮN NHÃN Popular, bao nhiêu cái đúng?"**
- Nhìn vào những gì model báo động (580 app bị gắn nhãn Popular) và hỏi: model báo có đúng không?
- Công thức: bắt đúng ÷ tổng số gắn nhãn Popular = 411 ÷ (411 + 169) = 411 ÷ 580 = **71%**
- Precision thấp nghĩa là model "hô hoán" nhầm nhiều — gắn nhãn hot cho những app thực ra không hot.

**Ẩn dụ — máy báo cháy:**
- **Recall** = trong tất cả các vụ cháy thực sự xảy ra, máy báo động kêu được bao nhiêu vụ? (Bỏ sót một vụ cháy = thảm họa.)
- **Precision** = trong tất cả các lần máy kêu, bao nhiêu lần là cháy thật? (Báo động giả = phiền, nhưng không nguy hiểm.)

**Trong trường hợp model này:** recall (62%) thấp hơn precision (71%) — model có xu hướng **bỏ sót** một app hot hơn là **báo nhầm** một app không hot thành hot. Nếu mục tiêu kinh doanh là không bỏ lỡ app tiềm năng (ví dụ để đầu tư quảng cáo sớm), nên cân nhắc đánh đổi bớt precision để lấy thêm recall — ví dụ bằng cách hạ ngưỡng quyết định (decision threshold) hoặc dùng `class_weight='balanced'`.

## Thử Với App Mới — Dự Đoán 1 App Chưa Từng Thấy

In [6]:
new_app_raw = pd.DataFrame([{
    "Category": "GAME",
    "Size_MB": 45.0,
    "Price_Num": 0.0,
    "Type": "Free",
    "Content Rating": "Everyone"
}])

new_app_raw

,Category,Size_MB,Price_Num,Type,Content Rating
0,GAME,45.0,0.0,Free,Everyone


In [7]:
# áp dụng đúng các bước biến đổi mà dữ liệu train đã trải qua
new_app = pd.get_dummies(new_app_raw, columns=["Category", "Type", "Content Rating"])

# căn chỉnh cột cho khớp X_train: thêm cột dummy còn thiếu (điền 0), bỏ/sắp lại cho đúng thứ tự
new_app = new_app.reindex(columns=X_train.columns, fill_value=0)

model.predict_proba(new_app)

array([[0.26909005, 0.73090995]])

Một app game miễn phí, 45MB, dành cho mọi lứa tuổi — mô tả theo đúng cách nó xuất hiện trong dữ liệu gốc (text thường, chưa phải dummy columns).

Để chấm điểm, ta cho nó đi qua **đúng 2 bước biến đổi** mà dữ liệu train đã trải qua:

1. **`pd.get_dummies(...)`** — biến `Category`, `Type`, `Content Rating` thành các cột 0/1, giống hệt Bước 2 đã làm cho toàn bộ tập train. Lưu ý ta không truyền `drop_first=True` ở đây — với 1 dòng, `get_dummies` chỉ tạo cột cho category thực sự xuất hiện, các category còn thiếu sẽ được xử lý ở bước tiếp theo.
2. **`.reindex(columns=X_train.columns, fill_value=0)`** — căn cột cho khớp chính xác với `X_train`: cột dummy nào model cần mà dòng này không có (vì category đó không xuất hiện) sẽ được điền `0`, và các cột được sắp lại đúng thứ tự như lúc train.

Cách này mô phỏng đúng việc chấm điểm 1 app mới trong thực tế: lấy đặc điểm thô của nó, rồi cho đi qua **đúng pipeline tiền xử lý** đã dùng lúc train — không bao giờ tự tay mã hóa dòng dữ liệu, vì dễ sai và không scale được.

`predict_proba` trả về xác suất đứng sau nhãn 0/1 — cột 1 là xác suất app này phổ biến.

## Recap — Toàn Bộ Model, ~12 Dòng Code

In [8]:
X = pd.get_dummies(df[["Category","Size_MB","Price_Num","Type","Content Rating"]],
                    columns=["Category","Type","Content Rating"], drop_first=True)
y = (df["Installs_Num"] >= 500_000).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=["Not Popular", "Popular"]))

              precision    recall  f1-score   support

 Not Popular       0.72      0.79      0.75       820
     Popular       0.71      0.62      0.66       665

    accuracy                           0.72      1485
   macro avg       0.71      0.71      0.71      1485
weighted avg       0.71      0.72      0.71      1485



Đây là của bạn rồi — đổi sang CSV khác, đổi target column khác, cùng đoạn code này vẫn chạy được.

## Nâng Cấp Model — Feature Engineering + Class Weight + Threshold

Model ở Bước 5 có Recall cho lớp Popular chỉ 62% — bỏ sót gần 4/10 app thực sự hot. Ba cải tiến dưới đây đều **không đụng đến Installs_Num hay Reviews** (tránh data leakage đã nói ở trên), chỉ thêm tín hiệu hợp lệ và điều chỉnh cách model ra quyết định.

**1. Feature mới:** độ dài tên app, số ngày kể từ lần cập nhật cuối (app cập nhật gần đây thường "sống khỏe" hơn)
**2. `class_weight='balanced'`:** phạt nặng hơn khi model đoán sai lớp Popular
**3. Chuẩn hóa (`StandardScaler`)** cho các cột số, vì Logistic Regression nhạy với thang đo khác nhau giữa Size_MB và Price_Num

In [9]:
df["App_Name_Length"] = df["App"].str.len()

ref_date = pd.to_datetime(df["Last_Updated_Date"]).max()
df["Days_Since_Update"] = (ref_date - pd.to_datetime(df["Last_Updated_Date"])).dt.days

features_v2 = ["Category", "Size_MB", "Price_Num", "Type", "Content Rating",
               "App_Name_Length", "Days_Since_Update"]
X2 = df[features_v2]

X2.head()

,Category,Size_MB,Price_Num,Type,Content Rating,App_Name_Length,Days_Since_Update
0,ART_AND_DESIGN,19.0,0.0,Free,Everyone,46,213
1,ART_AND_DESIGN,14.0,0.0,Free,Everyone,19,205
2,ART_AND_DESIGN,8.7,0.0,Free,Everyone,50,7
3,ART_AND_DESIGN,25.0,0.0,Free,Teen,21,61
4,ART_AND_DESIGN,2.8,0.0,Free,Everyone,37,49


In [10]:
X2 = pd.get_dummies(X2, columns=["Category", "Type", "Content Rating"], drop_first=True)

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y, test_size=0.2, random_state=42, stratify=y)

from sklearn.preprocessing import StandardScaler

numeric_cols = ["Size_MB", "Price_Num", "App_Name_Length", "Days_Since_Update"]
scaler = StandardScaler()
X2_train[numeric_cols] = scaler.fit_transform(X2_train[numeric_cols])
X2_test[numeric_cols] = scaler.transform(X2_test[numeric_cols])

X2_train.shape, X2_test.shape

((5939, 42), (1485, 42))

In [12]:
model_v2 = LogisticRegression(max_iter=1000, class_weight="balanced")
model_v2.fit(X2_train, y2_train)

probs_v2 = model_v2.predict_proba(X2_test)[:, 1]
y_pred_v2 = (probs_v2 >= 0.5).astype(int)   # vẫn dùng ngưỡng mặc định 0.5 để so sánh công bằng

print(confusion_matrix(y2_test, y_pred_v2))
print()
print(classification_report(y2_test, y_pred_v2, target_names=["Not Popular", "Popular"]))

[[585 235]
 [165 500]]

              precision    recall  f1-score   support

 Not Popular       0.78      0.71      0.75       820
     Popular       0.68      0.75      0.71       665

    accuracy                           0.73      1485
   macro avg       0.73      0.73      0.73      1485
weighted avg       0.74      0.73      0.73      1485



### Thử Thêm — Hạ Decision Threshold

`class_weight='balanced'` đã giúp nhiều, nhưng ta vẫn có thể vặn thêm bằng cách hạ ngưỡng quyết định — thay vì cần xác suất ≥ 50% mới gắn nhãn Popular, hạ xuống 40%:

In [14]:
y_pred_v2_low = (probs_v2 >= 0.4).astype(int)

print(confusion_matrix(y2_test, y_pred_v2_low))
print()
print(classification_report(y2_test, y_pred_v2_low, target_names=["Not Popular", "Popular"]))

[[450 370]
 [ 80 585]]

              precision    recall  f1-score   support

 Not Popular       0.85      0.55      0.67       820
     Popular       0.61      0.88      0.72       665

    accuracy                           0.70      1485
   macro avg       0.73      0.71      0.69      1485
weighted avg       0.74      0.70      0.69      1485



### So Sánh 3 Phiên Bản

| Phiên bản | Recall (Popular) | Precision (Popular) | Accuracy |
|---|---|---|---|
| Bước 5 — baseline gốc | 62% | 71% | 72% |
| v2 — thêm feature + class_weight (threshold 0.5) | 75% | 68% | 73% |
| v2 — threshold hạ xuống 0.4 | **88%** | 61% | 70% |

**Nhận xét:** feature engineering + `class_weight='balanced'` mang lại cải thiện *thật* — recall tăng từ 62% → 75% mà accuracy còn nhích lên (72% → 73%), không phải đánh đổi, mà là cải thiện thuần.

Hạ threshold thêm xuống 0.4 là một nút vặn *miễn phí* (không cần train lại model) đẩy recall lên tới 88%, nhưng lần này đánh đổi rõ rệt: accuracy giảm về 70%, precision rơi xuống 61% (61% app bị gắn nhãn Popular thực ra không phải).

Chọn threshold nào phụ thuộc mục tiêu thực tế: nếu bỏ sót 1 app hot tốn kém hơn nhiều so với báo nhầm 1 app không hot (ví dụ dùng để lọc app tiềm năng đầu tư quảng cáo sớm), threshold 0.4 hợp lý. Nếu cần độ tin cậy cao hơn khi gắn nhãn Popular, nên giữ ở 0.5.